# Пояснения к решению

Результаты:
1. Таблица output.xlsx
2. Дашборд  https://datalens.yandex/kpqcp1f3nb2c4

**Обработка данных**

**Дата** была обработана исходя из того, что возможны переходящие записи реестра с одного месяца на другой. В подобных случаях я обычно уточнях данные у операторов или подрядчиков. По умолчанию доверяю больше дате в файле чем дате в названии файла. Неявный формат даты может скрывать путаницу дня и месяца.

**Отсутствующие значения** были удалены так как невозможно установить объем и дату. Требует уточнения у вводившего данные или у подрядчика.\
**Категориальные значения** где надо были удалены пробелы, но в общем случае необходим подход со справочником

**Дашборд** реализован в Datalens на основании статичного файла output.xlsx. В зависимости от потребностей можно в дальнейшем еще больше автоматизировать процесс, предприняв следующие шаги
1. Сохранять скриптом данные в Google Spreadsheets как источник данных для Datalens
2. Или поднять на VPS сервер для сохранения данных в PostgresSQL как источник данных для Datalens
3. Или использовать на VPS дашборд на основании DataLens
4. Использовать streamlit или dash интерфейс

Любой из подходов я могу реализовать в зависимости от требований.
Одно ограничение - предпочитаю не использовать продукты Microsoft.

In [97]:
import os
import pandas as pd


In [83]:
data = {}

for filename in os.listdir('input'):
    name = filename.removeprefix('data_').removesuffix('.xlsx')
    data[name] = pd.read_excel(f'input/{filename}')
    data[name]['source'] = filename


In [84]:
for name, piece in data.items():
    print(name, piece.columns)


february Index(['date', 'constructor', 'work_type', 'volume', 'source'], dtype='object')
january Index(['Дата', 'Подрядчик', 'Тип_работы', 'Обьем', 'source'], dtype='object')


In [85]:
column_map = {
    'Дата': 'date',
    'Подрядчик': 'constructor',
    'Тип_работы': 'work_type',
    'Обьем': 'volume'
}
data['january'].rename(columns=column_map, inplace=True)
data['january'].columns


Index(['date', 'constructor', 'work_type', 'volume', 'source'], dtype='object')

In [86]:
data['february']


,date,constructor,work_type,volume,source
0,2024/02/01,ООО Ромашка,Монтаж,1500.0,data_february.xlsx
1,02-02-2024,ЗАО Лотос,Ремонт,2500.0,data_february.xlsx
2,2024-02-03,ЗАО Лотос,Ремонт,2500.0,data_february.xlsx
3,2024-02-04,ИП Иванов,Монтаж,1800.0,data_february.xlsx
4,NaN,ООО Ромашка,Монтаж,NaN,data_february.xlsx


In [87]:
data['january']


,date,constructor,work_type,volume,source
0,2024-01-01,ООО Ромашка,Монтаж,1000.0,data_january.xlsx
1,01/02/2024,ООО Ромашка,Монтаж,1000.0,data_january.xlsx
2,2024.01.03,ЗАО Лотос,Ремонт,2000.0,data_january.xlsx
3,NaN,ИП Иванов,Монтаж,NaN,data_january.xlsx
4,2024-01-05,ЗАО Лотос,Ремонт,2000.0,data_january.xlsx
5,01/02/2024,ООО Ромашка,Монтаж,1000.0,data_january.xlsx


In [88]:
def date_coercion_strategies(dates: pd.Series):
    dates = dates.apply(lambda x: str(x).replace('/', '.').replace('-', '.'))

    formats = ['%Y.%m.%d', '%d.%m.%Y', '%m.%d.%Y']

    for fmt in formats:
        mask = pd.to_datetime(dates, format=fmt, errors='coerce').notna()
        dates.loc[mask] = pd.to_datetime(dates.loc[mask], format=fmt).dt.date

    dates.loc[dates == 'nan'] = pd.NA
    return dates

for name, piece in data.items():
    data[name]['date'] = date_coercion_strategies(data[name]['date'])
    print(data[name]['date'])


0    2024-02-01
1    2024-02-02
2    2024-02-03
3    2024-02-04
4          <NA>
Name: date, dtype: object
0    2024-01-01
1    2024-02-01
2    2024-01-03
3          <NA>
4    2024-01-05
5    2024-02-01
Name: date, dtype: object


In [89]:
df = pd.concat([*data.values()])
df


,date,constructor,work_type,volume,source
0,2024-02-01,ООО Ромашка,Монтаж,1500.0,data_february.xlsx
1,2024-02-02,ЗАО Лотос,Ремонт,2500.0,data_february.xlsx
2,2024-02-03,ЗАО Лотос,Ремонт,2500.0,data_february.xlsx
3,2024-02-04,ИП Иванов,Монтаж,1800.0,data_february.xlsx
4,<NA>,ООО Ромашка,Монтаж,NaN,data_february.xlsx
0,2024-01-01,ООО Ромашка,Монтаж,1000.0,data_january.xlsx
1,2024-02-01,ООО Ромашка,Монтаж,1000.0,data_january.xlsx
2,2024-01-03,ЗАО Лотос,Ремонт,2000.0,data_january.xlsx
3,<NA>,ИП Иванов,Монтаж,NaN,data_january.xlsx
4,2024-01-05,ЗАО Лотос,Ремонт,2000.0,data_january.xlsx


In [90]:
print(df['constructor'].unique())
print(df['work_type'].unique())


['ООО Ромашка' 'ЗАО Лотос' 'ИП Иванов' ' ООО Ромашка ']
['Монтаж' 'Ремонт']


In [91]:
df['constructor'] = df['constructor'].apply(lambda x: x.strip())


In [92]:
print(df['constructor'].unique())


['ООО Ромашка' 'ЗАО Лотос' 'ИП Иванов']


Итоговая таблица

In [93]:
df.columns


Index(['date', 'constructor', 'work_type', 'volume', 'source'], dtype='object')

In [94]:
final_df = (
    df
    .sort_values(by='date', na_position='last')
    .dropna()
    .rename(
        columns={
            'date': 'Дата',
            'constructor': 'Подрядчик',
            'work_type': 'Тип работы',
            'volume': 'Объем',
            'source': 'Источник',
        }
    )
)
final_df


,Дата,Подрядчик,Тип работы,Объем,Источник
0,2024-01-01,ООО Ромашка,Монтаж,1000.0,data_january.xlsx
2,2024-01-03,ЗАО Лотос,Ремонт,2000.0,data_january.xlsx
4,2024-01-05,ЗАО Лотос,Ремонт,2000.0,data_january.xlsx
0,2024-02-01,ООО Ромашка,Монтаж,1500.0,data_february.xlsx
1,2024-02-01,ООО Ромашка,Монтаж,1000.0,data_january.xlsx
5,2024-02-01,ООО Ромашка,Монтаж,1000.0,data_january.xlsx
1,2024-02-02,ЗАО Лотос,Ремонт,2500.0,data_february.xlsx
2,2024-02-03,ЗАО Лотос,Ремонт,2500.0,data_february.xlsx
3,2024-02-04,ИП Иванов,Монтаж,1800.0,data_february.xlsx


In [96]:
final_df.to_excel('output.xlsx', index=False)
